# 8A - Train Saved LightGBM Cluster Labeler

This notebook trains and saves the production LightGBM labeler from the fixed HDBSCAN cluster output.

It is part of Pipeline A because the model must be trained after the full clustering pipeline is complete.

Default mode in this A copy:

```python
TRAIN_MODE = True
SCORE_MODE = False
```


In [ ]:
import json
import warnings
from datetime import datetime
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Configuration

Change this cell depending on the workflow.

- `TRAIN_MODE = True`: train and save LightGBM models from current HDBSCAN labels.
- `SCORE_MODE = True`: load saved models and score a new incoming developer feature table.

For the first run, keep both as `True`. For future scoring-only runs, set `TRAIN_MODE = False` and `SCORE_MODE = True`.


In [ ]:
# ---------------------------------------------------------------------
# Main switches
# ---------------------------------------------------------------------
TRAIN_MODE = True
SCORE_MODE = False

# ---------------------------------------------------------------------
# DuckDB database
# ---------------------------------------------------------------------
DB_PATH = "developer_project.duckdb"
ID_COL = "developer_id"

# Tables from the current project pipeline
TRAIN_PROFILE_TABLE = "dev_profile_final_v4"
SOURCE_CLUSTER_TABLE = "dev_lifecycle_cluster_membership_v11_final"

# New incoming developer feature table
# This should be created by running the same cleaning + feature engineering logic
# on the new raw data batch.
NEW_PROFILE_TABLE = "dev_profile_new_incoming_v1"

# Output tables
SCORED_NEW_TABLE = "dev_new_developer_cluster_scores_v1"
TRAIN_PRED_TABLE = "dev_saved_labeler_training_predictions_v1"
TRAIN_STATS_TABLE = "dev_saved_labeler_training_stats_v1"

# Saved model artifacts
ARTIFACT_DIR = Path("supervised_cluster_labeler_artifacts_v1")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Training controls
MAX_TRAIN_ROWS_PER_STRATUM = 250_000
TEST_SIZE = 0.20
INCLUDE_NOISE_AS_CLASS = True

# These are scored with LightGBM
MODEL_STRATA = ["active", "cooling", "at_risk"]

# These are usually assigned by business rules, not by the model
RULE_CARRY_FORWARD_STRATA = ["dormant", "unactivated", "unknown"]

con = duckdb.connect(DB_PATH)
print("Connected to:", DB_PATH)
print("Artifact directory:", ARTIFACT_DIR.resolve())


## 2. Feature Set

These features should match the HDBSCAN V11 feature logic. Do not change them unless the feature engineering notebook changes and you plan to retrain the saved model.


In [ ]:
FEATURES_BY_STRATUM = {
    "active": [
        "log_activity_count_0_30d",
        "log_activity_count_30_90d",
        "unique_activity_types_0_30d",
        "unique_modalities_0_30d",
        "developer_effort_score",
        "weighted_recent_confidence_effort",
        "recent_build_flag",
        "log_build_count_0_30d",
        "build_share_lifetime",
        "activity_velocity_0_30_vs_30_90",
        "log_clipped_lifetime_activity_count_p99",
        "persona_entropy",
    ],
    "cooling": [
        "log_activity_count_30_90d",
        "log_activity_count_90_180d",
        "developer_effort_score",
        "weighted_recent_confidence_effort",
        "build_share_lifetime",
        "high_effort_share_lifetime",
        "log_clipped_lifetime_activity_count_p99",
        "persona_entropy",
    ],
    "at_risk": [
        "log_activity_count_30_90d",
        "log_activity_count_90_180d",
        "developer_effort_score",
        "build_share_lifetime",
        "high_effort_share_lifetime",
        "log_clipped_lifetime_activity_count_p99",
        "persona_entropy",
    ],
}

ALL_MODEL_FEATURES = sorted({f for features in FEATURES_BY_STRATUM.values() for f in features})
print("Total unique model features:", len(ALL_MODEL_FEATURES))


## 3. Helper Functions

In [ ]:
def table_exists(con, table_name: str) -> bool:
    sql = """
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_name = ?
    """
    return con.execute(sql, [table_name]).fetchone()[0] > 0


def get_columns(con, table_name: str) -> list[str]:
    return con.execute(f"DESCRIBE {table_name}").df()["column_name"].astype(str).tolist()


def require_tables(*table_names):
    missing = [t for t in table_names if not table_exists(con, t)]
    if missing:
        raise ValueError(f"Missing required DuckDB tables: {missing}")


def save_df_to_table(con, df: pd.DataFrame, table_name: str):
    view_name = f"tmp_{table_name}"
    con.register(view_name, df)
    con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM {view_name}")
    con.unregister(view_name)


def clean_feature_frame(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in features:
        if c not in out.columns:
            out[c] = np.nan
    return out[features].replace([np.inf, -np.inf], np.nan)


def stratified_train_sample(df: pd.DataFrame, label_col: str, max_rows: int) -> pd.DataFrame:
    if len(df) <= max_rows:
        return df.copy()
    pieces = []
    counts = df[label_col].value_counts(dropna=False)
    for label, n in counts.items():
        frac = n / len(df)
        take = max(100, int(round(frac * max_rows)))
        take = min(take, n)
        pieces.append(df[df[label_col] == label].sample(n=take, random_state=RANDOM_STATE))
    sampled = pd.concat(pieces, ignore_index=True)
    if len(sampled) > max_rows:
        sampled = sampled.sample(n=max_rows, random_state=RANDOM_STATE).reset_index(drop=True)
    return sampled


def make_preprocessor() -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])


def load_lightgbm_classifier(n_classes: int):
    try:
        from lightgbm import LGBMClassifier
    except Exception as exc:
        raise ImportError("LightGBM is required. Install with: pip install lightgbm") from exc

    return LGBMClassifier(
        objective="multiclass" if n_classes > 2 else "binary",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=64,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )


def artifact_path(stratum: str) -> Path:
    return ARTIFACT_DIR / f"cluster_labeler_lgbm_{stratum}.joblib"


def metadata_path() -> Path:
    return ARTIFACT_DIR / "cluster_labeler_metadata.json"


## 4. Train and Save LightGBM Labelers

This trains one classifier per lifecycle stratum. That is better than one global model because active, cooling, and at-risk developers have different behavioral meanings.


In [ ]:
def load_training_frame(stratum: str, features: list[str]) -> pd.DataFrame:
    available_features = [f for f in features if f in get_columns(con, TRAIN_PROFILE_TABLE)]
    select_profile_cols = [ID_COL] + available_features
    select_sql = ", ".join([f"p.{c}" for c in select_profile_cols])
    noise_filter = "" if INCLUDE_NOISE_AS_CLASS else "AND LOWER(CAST(m.cluster_key AS VARCHAR)) NOT LIKE '%noise%'"

    sql = f"""
        SELECT
            {select_sql},
            LOWER(CAST(m.stratum AS VARCHAR)) AS stratum,
            CAST(m.cluster_key AS VARCHAR) AS source_cluster_key
        FROM {SOURCE_CLUSTER_TABLE} m
        JOIN {TRAIN_PROFILE_TABLE} p USING ({ID_COL})
        WHERE LOWER(CAST(m.stratum AS VARCHAR)) = '{stratum}'
          {noise_filter}
    """
    df = con.execute(sql).df()
    for f in features:
        if f not in df.columns:
            df[f] = np.nan
    return df[[ID_COL] + features + ["stratum", "source_cluster_key"]]


def train_one_stratum(stratum: str) -> dict:
    features = FEATURES_BY_STRATUM[stratum]
    df = load_training_frame(stratum, features)
    if df.empty:
        raise ValueError(f"No training rows found for stratum: {stratum}")

    print("\n" + "=" * 80)
    print(f"Training LightGBM labeler for stratum: {stratum}")
    print("Full labeled rows:", len(df))
    display(df["source_cluster_key"].value_counts().reset_index(name="n"))

    train_df = stratified_train_sample(df, label_col="source_cluster_key", max_rows=MAX_TRAIN_ROWS_PER_STRATUM)
    print("Training sample rows:", len(train_df))

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(train_df["source_cluster_key"].astype(str))
    n_classes = len(label_encoder.classes_)
    if n_classes < 2:
        raise ValueError(f"Need at least two cluster classes for {stratum}; found {n_classes}")

    X = clean_feature_frame(train_df, features)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    preprocessor = make_preprocessor()
    X_train_t = preprocessor.fit_transform(X_train)
    X_test_t = preprocessor.transform(X_test)

    model = load_lightgbm_classifier(n_classes=n_classes)
    model.fit(X_train_t, y_train)

    pred = model.predict(X_test_t)
    metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
        "macro_f1": float(f1_score(y_test, pred, average="macro")),
        "weighted_f1": float(f1_score(y_test, pred, average="weighted")),
        "n_train_rows": int(len(train_df)),
        "n_full_rows": int(len(df)),
        "n_classes": int(n_classes),
    }

    print("Metrics:", metrics)
    print(classification_report(y_test, pred, target_names=label_encoder.classes_))

    bundle = {
        "model_type": "lightgbm",
        "model": model,
        "preprocessor": preprocessor,
        "label_encoder": label_encoder,
        "features": features,
        "stratum": stratum,
        "metrics": metrics,
        "classes": label_encoder.classes_.tolist(),
        "created_at": datetime.utcnow().isoformat() + "Z",
        "train_profile_table": TRAIN_PROFILE_TABLE,
        "source_cluster_table": SOURCE_CLUSTER_TABLE,
        "include_noise_as_class": INCLUDE_NOISE_AS_CLASS,
    }

    joblib.dump(bundle, artifact_path(stratum))
    print("Saved artifact:", artifact_path(stratum))
    return bundle


if TRAIN_MODE:
    require_tables(TRAIN_PROFILE_TABLE, SOURCE_CLUSTER_TABLE)

    stats_rows = []
    for stratum in MODEL_STRATA:
        bundle = train_one_stratum(stratum)
        stats_rows.append({
            "stratum": stratum,
            "model_type": bundle["model_type"],
            **bundle["metrics"],
            "artifact_path": str(artifact_path(stratum)),
            "created_at": bundle["created_at"],
        })

    stats_df = pd.DataFrame(stats_rows)
    save_df_to_table(con, stats_df, TRAIN_STATS_TABLE)
    display(stats_df)

    metadata = {
        "model_type": "lightgbm",
        "model_strata": MODEL_STRATA,
        "rule_carry_forward_strata": RULE_CARRY_FORWARD_STRATA,
        "features_by_stratum": FEATURES_BY_STRATUM,
        "artifact_dir": str(ARTIFACT_DIR),
        "created_at": datetime.utcnow().isoformat() + "Z",
        "train_profile_table": TRAIN_PROFILE_TABLE,
        "source_cluster_table": SOURCE_CLUSTER_TABLE,
        "include_noise_as_class": INCLUDE_NOISE_AS_CLASS,
    }
    metadata_path().write_text(json.dumps(metadata, indent=2))
    print("Saved metadata:", metadata_path())


## 5. Audit Predictions on the Training Population

This table checks whether the saved classifier reproduces the original HDBSCAN label structure closely enough to be used as a scoring layer.


In [ ]:
def predict_with_bundle(df: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    features = bundle["features"]
    X = clean_feature_frame(df, features)
    Xt = bundle["preprocessor"].transform(X)

    pred_num = bundle["model"].predict(Xt)
    pred_label = bundle["label_encoder"].inverse_transform(pred_num.astype(int))

    if hasattr(bundle["model"], "predict_proba"):
        proba = bundle["model"].predict_proba(Xt)
        confidence = proba.max(axis=1)
        entropy = -(proba * np.log(np.clip(proba, 1e-12, 1))).sum(axis=1)
    else:
        confidence = np.repeat(np.nan, len(df))
        entropy = np.repeat(np.nan, len(df))

    out = df[[ID_COL]].copy()
    out["stratum"] = bundle["stratum"]
    out["predicted_cluster_key"] = pred_label
    out["prediction_confidence"] = confidence
    out["prediction_entropy"] = entropy
    out["assignment_source"] = "saved_lightgbm_labeler"
    return out


if TRAIN_MODE:
    audit_parts = []
    for stratum in MODEL_STRATA:
        bundle = joblib.load(artifact_path(stratum))
        df = load_training_frame(stratum, bundle["features"])
        pred_df = predict_with_bundle(df, bundle)
        actual_df = df[[ID_COL, "source_cluster_key"]].rename(columns={"source_cluster_key": "actual_hdbscan_cluster_key"})
        audit_parts.append(pred_df.merge(actual_df, on=ID_COL, how="left"))

    train_pred_df = pd.concat(audit_parts, ignore_index=True)
    train_pred_df["matches_original_hdbscan_label"] = (
        train_pred_df["predicted_cluster_key"].astype(str)
        == train_pred_df["actual_hdbscan_cluster_key"].astype(str)
    )

    save_df_to_table(con, train_pred_df, TRAIN_PRED_TABLE)
    print(f"Saved training prediction audit table: {TRAIN_PRED_TABLE}")
    display(train_pred_df.groupby(["stratum", "matches_original_hdbscan_label"]).size().reset_index(name="n"))


## 6. Score New Incoming Developers

The new data may arrive in the same raw format as the original data. In that case, first run the same cleaning and feature engineering logic on the new raw batch to create `NEW_PROFILE_TABLE`. Then this section assigns existing cluster labels.

Important: this notebook does **not** rediscover new clusters. It scores new developers into the existing cluster framework.


In [ ]:
def load_new_stratum(stratum: str, features: list[str]) -> pd.DataFrame:
    new_cols = get_columns(con, NEW_PROFILE_TABLE)
    select_cols = [ID_COL, "stratum"] + [f for f in features if f in new_cols]
    select_sql = ", ".join(select_cols)
    df = con.execute(f"""
        SELECT {select_sql}
        FROM {NEW_PROFILE_TABLE}
        WHERE LOWER(CAST(stratum AS VARCHAR)) = '{stratum}'
    """).df()
    for f in features:
        if f not in df.columns:
            df[f] = np.nan
    return df[[ID_COL, "stratum"] + features]


def score_new_developers() -> pd.DataFrame:
    scored_parts = []

    for stratum in MODEL_STRATA:
        bundle = joblib.load(artifact_path(stratum))
        new_df = load_new_stratum(stratum, bundle["features"])
        if new_df.empty:
            print(f"No new rows for model stratum: {stratum}")
            continue
        scored = predict_with_bundle(new_df, bundle)
        scored_parts.append(scored)
        print(f"Scored {len(scored):,} new developers for stratum: {stratum}")

    new_cols = get_columns(con, NEW_PROFILE_TABLE)
    rule_list = ", ".join([repr(s) for s in RULE_CARRY_FORWARD_STRATA])
    if "cluster_key" in new_cols:
        carry_sql = f"""
            SELECT
                {ID_COL},
                LOWER(CAST(stratum AS VARCHAR)) AS stratum,
                CAST(cluster_key AS VARCHAR) AS predicted_cluster_key,
                CAST(1.0 AS DOUBLE) AS prediction_confidence,
                CAST(0.0 AS DOUBLE) AS prediction_entropy,
                'rule_based_carry_forward' AS assignment_source
            FROM {NEW_PROFILE_TABLE}
            WHERE LOWER(CAST(stratum AS VARCHAR)) IN ({rule_list})
        """
    else:
        carry_sql = f"""
            SELECT
                {ID_COL},
                LOWER(CAST(stratum AS VARCHAR)) AS stratum,
                CASE
                    WHEN LOWER(CAST(stratum AS VARCHAR)) = 'dormant' THEN 'Dormant_Rule_Based'
                    WHEN LOWER(CAST(stratum AS VARCHAR)) = 'unactivated' THEN 'Unactivated'
                    ELSE 'Unknown'
                END AS predicted_cluster_key,
                CAST(1.0 AS DOUBLE) AS prediction_confidence,
                CAST(0.0 AS DOUBLE) AS prediction_entropy,
                'rule_based_default' AS assignment_source
            FROM {NEW_PROFILE_TABLE}
            WHERE LOWER(CAST(stratum AS VARCHAR)) IN ({rule_list})
        """

    carry_df = con.execute(carry_sql).df()
    if not carry_df.empty:
        scored_parts.append(carry_df)
        print(f"Carried forward {len(carry_df):,} rule-based developers")

    if not scored_parts:
        raise ValueError("No developers were scored. Check NEW_PROFILE_TABLE and stratum values.")

    out = pd.concat(scored_parts, ignore_index=True)
    out["scored_at"] = datetime.utcnow().isoformat() + "Z"
    out["model_artifact_dir"] = str(ARTIFACT_DIR)
    return out


if SCORE_MODE:
    require_tables(NEW_PROFILE_TABLE)
    missing_artifacts = [str(artifact_path(s)) for s in MODEL_STRATA if not artifact_path(s).exists()]
    if missing_artifacts:
        raise ValueError("Missing saved model artifacts. Run TRAIN_MODE=True first. Missing: " + ", ".join(missing_artifacts))

    scored_new_df = score_new_developers()
    save_df_to_table(con, scored_new_df, SCORED_NEW_TABLE)
    print(f"Saved scored new developer table: {SCORED_NEW_TABLE}")
    print("Rows:", len(scored_new_df))
    display(scored_new_df.groupby(["stratum", "predicted_cluster_key", "assignment_source"]).size().reset_index(name="n").head(100))


## 7. Export Results

Exports scored results and model metadata. The `.joblib` model files are the key artifacts for future scoring.


In [ ]:
EXPORT_DIR = Path("toexport_new_developer_scoring")
EXPORT_DIR.mkdir(exist_ok=True)

if table_exists(con, SCORED_NEW_TABLE):
    out_path = EXPORT_DIR / f"{SCORED_NEW_TABLE}.parquet"
    con.execute(f"COPY {SCORED_NEW_TABLE} TO '{out_path.as_posix()}' (FORMAT PARQUET)")
    print("Exported scored new developer table:", out_path)

if table_exists(con, TRAIN_STATS_TABLE):
    stats_path = EXPORT_DIR / f"{TRAIN_STATS_TABLE}.parquet"
    con.execute(f"COPY {TRAIN_STATS_TABLE} TO '{stats_path.as_posix()}' (FORMAT PARQUET)")
    print("Exported training stats table:", stats_path)

print("Saved model artifacts:")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print("-", p)


## 8. Two-Pipeline Operating Model

### Pipeline A: Current project / reclustering pipeline

Use when you need to rebuild the entire project from raw data or refresh cluster discovery.

1. `01_Creating_duckDB.ipynb`
2. `02_Cleaning.ipynb`
3. `03_FeatureEngineering_v3.ipynb`
4. `04_clustering_hdbscan.ipynb`
5. `05_GMM_Cluster_Stratified.ipynb` for validation
6. `06_hmm_categorical.ipynb` for journey sequences
7. `07_SupervisedClusterLabeling_LGBM_XGB.ipynb` or this notebook for supervised labeler training

### Pipeline B: New incoming raw data / fixed-cluster scoring pipeline

Use when new raw data arrives in the same format and the goal is to place developers into the existing cluster framework.

1. Load new raw data into a separate raw/staging schema or batch tables.
2. Apply the same cleaning logic as `02_Cleaning.ipynb`.
3. Apply the same feature engineering logic as `03_FeatureEngineering_v3.ipynb` to create `dev_profile_new_incoming_v1`.
4. Run this notebook with `TRAIN_MODE = False` and `SCORE_MODE = True`.
5. Use `dev_new_developer_cluster_scores_v1` for dashboards, recommendations, and monitoring.

### When to recluster instead of only scoring

Recluster when:

- new data sources are added,
- feature definitions change,
- large behavior shifts appear,
- new cluster types are suspected,
- prediction confidence drops for many new developers,
- business stakeholders want a refreshed segmentation.


In [ ]:
con.close()
print("DuckDB connection closed. Notebook complete.")
